# Notebook 3: Exploratory Data Analysis

## Purpose
General EDA to understand category & index distributions.

## Visuals
- Heatmaps of category proportions (grouped by Top/Mid/Trash)
- Distribution of indices (histograms, KDE)
- UMAP/TSNE of books by category proportions (colored by group)
- Correlation matrix of indices

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Visualization settings
sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 300

np.random.seed(42)

## 1. Load Data

In [ ]:
PROJECT_ROOT = Path().resolve().parent.parent.parent.parent
INPUT_FILE = PROJECT_ROOT / "results" / "stage10_correlation_analysis" / "statistical_analysis" / "indices_book.csv"
OUTPUT_DIR = PROJECT_ROOT / "results" / "stage10_correlation_analysis" / "statistical_analysis" / "eda"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(INPUT_FILE)
print(f"Loaded {len(df)} books")

## 2. Heatmaps of Category Proportions by Group

In [ ]:
# TODO: Create heatmap of category proportions grouped by Top/Mid/Trash
# Identify category columns (exclude metadata and index columns)
index_cols = ['love_over_sex', 'hea_index', 'luxury_x_love', 'protective_minus_jealous', 
              'dark_vs_tender', 'miscommunication_balance']
metadata_cols = ['book_id', 'group', 'average_rating_weighted_mean']
category_cols = [col for col in df.columns if col not in index_cols + metadata_cols]

if 'group' in df.columns and len(category_cols) > 0:
    # Compute mean proportions per group
    group_means = df.groupby('group')[category_cols].mean()
    
    # Create heatmap
    plt.figure(figsize=(12, 6))
    sns.heatmap(group_means.T, annot=True, fmt='.3f', cmap='viridis', cbar_kws={'label': 'Mean Proportion'})
    plt.title('Category Proportions by Popularity Group')
    plt.ylabel('Category')
    plt.xlabel('Group')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'category_heatmap_by_group.png')
    plt.show()

## 3. Distribution of Indices

In [ ]:
# Histograms and KDE plots for each index
if 'group' in df.columns:
    for idx in index_cols:
        if idx in df.columns:
            fig, axes = plt.subplots(1, 2, figsize=(12, 4))
            
            # Histogram
            axes[0].hist(df[idx], bins=30, alpha=0.7, edgecolor='black')
            axes[0].set_title(f'{idx} - Histogram')
            axes[0].set_xlabel('Value')
            axes[0].set_ylabel('Frequency')
            
            # KDE by group
            for group in df['group'].unique():
                subset = df[df['group'] == group][idx]
                sns.kdeplot(subset, label=group, ax=axes[1])
            axes[1].set_title(f'{idx} - KDE by Group')
            axes[1].set_xlabel('Value')
            axes[1].set_ylabel('Density')
            axes[1].legend()
            
            plt.tight_layout()
            plt.savefig(OUTPUT_DIR / f'{idx}_distribution.png')
            plt.show()

## 4. UMAP/TSNE Visualization

In [ ]:
# Dimensionality reduction of books by category proportions
try:
    from umap import UMAP
    from sklearn.preprocessing import StandardScaler
    
    if len(category_cols) > 0:
        # Prepare data
        X = df[category_cols].fillna(0).values
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        
        # UMAP
        umap_model = UMAP(n_components=2, random_state=42)
        X_umap = umap_model.fit_transform(X_scaled)
        
        # Plot
        plt.figure(figsize=(10, 8))
        if 'group' in df.columns:
            for group in df['group'].unique():
                mask = df['group'] == group
                plt.scatter(X_umap[mask, 0], X_umap[mask, 1], label=group, alpha=0.6)
            plt.legend()
        else:
            plt.scatter(X_umap[:, 0], X_umap[:, 1], alpha=0.6)
        plt.title('UMAP Visualization of Books by Category Proportions')
        plt.xlabel('UMAP 1')
        plt.ylabel('UMAP 2')
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / 'umap_books_by_categories.png')
        plt.show()
except ImportError:
    print("UMAP not available, skipping...")

## 5. Correlation Matrix of Indices

In [ ]:
# Correlation matrix
available_indices = [idx for idx in index_cols if idx in df.columns]
if len(available_indices) > 1:
    corr_matrix = df[available_indices].corr()
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
                square=True, linewidths=1, cbar_kws={'label': 'Correlation'})
    plt.title('Correlation Matrix of Indices')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'index_correlation_matrix.png')
    plt.show()

## 6. Summary Statistics

In [ ]:
# Summary statistics
summary_stats = df[available_indices].describe()
print("Summary Statistics for Indices:")
print(summary_stats)

# Save
summary_stats.to_csv(OUTPUT_DIR / 'summary_statistics.csv')
print(f"\n✓ Saved summary statistics to {OUTPUT_DIR / 'summary_statistics.csv'}")

## Summary

EDA complete. Insights documented. Next: Notebook 4 (Group Comparisons)